In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import zscore

# Chargement des données
data = pd.read_csv("listings.csv")
print(data.head())

# Suppression des outliers (z-score > 3 sur 'price')
data = data[(np.abs(zscore(data['price'])) < 3)]



Top 5 des configurations testées :
     mean_test_score  std_test_score  param_regressor__C  \
66         -0.374055        0.005395                 1.0   
60         -0.375257        0.005913                 1.0   
72         -0.375623        0.004504                 1.0   
129        -0.384205        0.006287               100.0   
99         -0.384552        0.004383                10.0   

     param_regressor__epsilon param_regressor__kernel param_regressor__gamma  
66                       0.10                     rbf                  scale  
60                       0.01                     rbf                  scale  
72                       0.20                     rbf                  scale  
129                      0.10                     rbf                   auto  
99                       0.10                     rbf                   auto  

MAE: 53.81
RMSE: 115.87
R²: 0.23

Prix prédit pour la nouvelle annonce : 122.20 $ / 113.65 €



Ce code commence par importer les bibliothèques nécessaires pour la manipulation des données, le traitement statistique et la modélisation avec scikit-learn. Il charge ensuite un jeu de données à partir d’un fichier CSV nommé "listings.csv" puis affiche les premières lignes pour avoir un aperçu. Pour améliorer la qualité des données, il supprime les valeurs aberrantes dans la colonne des prix en calculant le score z (z-score) de chaque valeur et en ne conservant que celles dont la valeur absolue est inférieure à 3. Cette étape permet d’éliminer les prix extrêmes qui pourraient fausser l’analyse et la modélisation ultérieure. Enfin, il affiche le nombre d’observations restantes après ce nettoyage.

Veux-tu que je te fasse un autre bloc explicatif pour la suite ?


In [ ]:

# Log-transformation des colonnes skewées
skewed_cols = ['minimum_nights', 'number_of_reviews', 'reviews_per_month']
for col in skewed_cols:
    if col in data.columns:
        data[col] = data[col].apply(lambda x: np.log1p(x))

# Cible log-transformée
y = np.log1p(data['price'])
X = data.drop(columns=['price'])

# Suppression des colonnes inutiles
for col in ['name', 'last_review', 'host_name', 'neighbourhood_group']:
    if col in X.columns:
        X.drop(columns=col, inplace=True)

# Colonnes numériques et catégorielles
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()


Ce bloc commence par appliquer une transformation logarithmique (`log1p`) aux colonnes fortement asymétriques (`minimum_nights`, `number_of_reviews`, `reviews_per_month`) afin de réduire l’effet des valeurs extrêmes et de rendre la distribution plus normale, ce qui facilite l’apprentissage du modèle.

La variable cible (`y`) correspond au prix transformé en log pour stabiliser la variance et mieux modéliser la relation. Les variables explicatives (`X`) sont obtenues en supprimant la colonne originale `price`.

Ensuite, plusieurs colonnes peu informatives ou redondantes (`name`, `last_review`, `host_name`, `neighbourhood_group`) sont supprimées pour éviter de polluer le modèle avec des données non pertinentes.

Enfin, le code identifie automatiquement les colonnes numériques et catégorielles dans le jeu de données, ce qui permettra d’appliquer un traitement adapté à chaque type de variable lors des étapes suivantes (prétraitement et encodage).

In [ ]:

# Prétraitement
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Pipeline complet
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(score_func=f_regression, k='all')),
    ('regressor', SVR())
])

# Grille d'hyperparamètres
param_grid = {
    'regressor__C': [0.01, 0.1, 1, 10, 100],
    'regressor__epsilon': [0.01, 0.1, 0.2, 0.5, 1],
    'regressor__kernel': ['rbf', 'linear', 'poly'],
    'regressor__gamma': ['scale', 'auto']
}


Ce code met en place un pipeline complet de prétraitement et de modélisation pour des données tabulaires.

Pour les variables numériques, il applique d'abord une imputation par la médiane pour gérer les valeurs manquantes, puis standardise les données avec un StandardScaler afin de centrer et réduire les variables.

Pour les variables catégorielles, il remplace les valeurs manquantes par la modalité la plus fréquente, puis encode ces variables en variables indicatrices (one-hot encoding) tout en ignorant les catégories inconnues lors de la prédiction.

Ces transformations sont combinées avec un ColumnTransformer qui applique automatiquement le traitement adapté selon le type de variable.

Ensuite, un pipeline global assemble ce prétraitement, une étape de sélection de caractéristiques avec SelectKBest (basée sur un test de régression), puis un modèle de régression par machines à vecteurs de support (SVR).

Une grille d’hyperparamètres est définie pour optimiser le modèle SVR via une recherche, incluant la régularisation C, la marge d’erreur epsilon, le type de noyau (rbf, linear, poly) et le paramètre gamma contrôlant la portée de l’influence des points de données.

In [ ]:

# Séparation train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Validation croisée
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Affichage des résultats de la validation croisée
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results = cv_results.sort_values(by="mean_test_score", ascending=False)

print("\nTop 5 des configurations testées :")
print(cv_results[[
    'mean_test_score', 'std_test_score',
    'param_regressor__C',
    'param_regressor__epsilon',
    'param_regressor__kernel',
    'param_regressor__gamma'
]].head(5))

# Meilleur modèle
best_model = grid_search.best_estimator_

# Évaluation
y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_exp = np.expm1(y_test)

mae = mean_absolute_error(y_test_exp, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_exp, y_pred))
r2 = r2_score(y_test_exp, y_pred)

print(f"\nMAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.2f}")

# Prédiction sur une annonce
sample = X_test.iloc[[0]]
predicted_log_price = best_model.predict(sample)
predicted_price_usd = np.expm1(predicted_log_price[0])
predicted_price_eur = predicted_price_usd * 0.93

print(f"\nPrix prédit pour la nouvelle annonce : {predicted_price_usd:.2f} $ / {predicted_price_eur:.2f} €")


Ce code définit un pipeline complet de traitement et de modélisation adapté à des données mixtes (numériques et catégorielles).

La transformation des variables numériques comprend une imputation des valeurs manquantes par la médiane, suivie d’une standardisation (centrage et réduction) grâce à StandardScaler.

Pour les variables catégorielles, les valeurs manquantes sont remplacées par la modalité la plus fréquente, puis elles sont converties en variables indicatrices binaires via un encodage one-hot, en ignorant les catégories inconnues lors de la phase de test.

Ces deux types de transformations sont combinés via un ColumnTransformer qui applique automatiquement la bonne transformation selon le type de variable.

Le pipeline complet enchaîne ce prétraitement, une étape de sélection des meilleures caractéristiques avec SelectKBest basée sur la relation linéaire avec la cible (f_regression), et un modèle de régression à support vectoriel (SVR).

Enfin, une grille d’hyperparamètres est préparée pour optimiser le modèle SVR. Elle explore différentes valeurs de la régularisation C, du paramètre d’insensibilité epsilon, des types de noyau (rbf, linear, poly), et des modes de calcul de gamma (scale ou auto).

## NOTES ##

Le meilleur MAE moyen obtenu est de 0.374 sur les données log-transformées (base e), avec un écart type faible (\~0.005) indiquant une bonne stabilité du modèle durant la validation croisée. Cependant, l’évaluation sur l’ensemble de test en prix réels montre une erreur moyenne absolue (MAE) d’environ 53.81 €, une RMSE de 115.87 € indiquant une erreur plus sévère sur les grandes valeurs, et un R² de seulement 0.23, ce qui signifie que le modèle n’explique que 23 % de la variance des prix réels. Cette différence traduit une perte de performance importante entre la validation croisée sur données transformées et le test final sur prix réels. Parmi les limites du modèle actuel, la transformation logarithmique bien que stabilisant la variance, rend les erreurs finales difficiles à interpréter directement. Le SVM montre une performance limitée avec de grands jeux de données hétérogènes, notamment à cause de l’encodage OneHot des variables catégorielles qui augmente la dimension, entraînant un temps de calcul élevé pour la grille SVR. Le score R² faible indique un sous-apprentissage possible, avec une incapacité à bien capturer les facteurs influents et les interactions non linéaires entre variables, surtout en présence de bruit ou peu de tuning. De plus, les données sont potentiellement déséquilibrées ou bruitées, les prix Airbnb variant beaucoup selon des critères qualitatifs (photos, description, réputation). Pour améliorer ce modèle, il serait pertinent de tester des modèles plus robustes comme Random Forest, Gradient Boosting (XGBoost, LightGBM, HistGradientBoostingRegressor) qui gèrent mieux les interactions et les données hétérogènes, d’appliquer une réduction de dimension (PCA) ou une sélection automatique de variables pour limiter la complexité due à OneHotEncoder, d’enrichir le feature engineering avec des variables comme le score d’évaluation par nuit ou le taux d’occupation, d’utiliser un encodage catégoriel avancé (Target Encoding) au lieu de OneHot pour les grosses colonnes, et d’augmenter le nombre de folds en validation croisée (5 ou 10) pour une meilleure estimation de la variance. L’exploration de modèles basés sur le deep learning pourrait aussi être envisagée si les données et ressources le permettent. En conclusion, le SVM testé, bien que stable en validation croisée, n’offre pas des performances satisfaisantes en production sur les prix Airbnb réels. L’erreur moyenne de 54 € reste élevée et le faible R² montre que le modèle ne capture pas la complexité des prix. Des modèles ensemblistes plus adaptés pourraient significativement améliorer la qualité prédictive tout en conservant une bonne robustesse face au bruit des données.
